# 09 - BLAST Search

This notebook demonstrates how to use the BLAST search functionality in PBI to search unknown sequences against the database.

## Prerequisites

- BLAST databases must be built first (run the pipeline with BLAST rules)
- BLAST+ must be installed in the environment

In [ ]:
from pbi import BlastSearcher

# Initialize with default data path (or provide explicit path)
searcher = BlastSearcher()
print("BlastSearcher initialized")

## Check Available Databases

In [ ]:
databases = searcher.list_databases()
for name, info in databases.items():
    status = "READY" if info["exists"] else "NOT BUILT"
    print(f"  [{status}] {name} ({info['type']})")

## Search a Nucleotide Sequence

Search an unknown DNA sequence against phage genomes.

In [ ]:
# Example: search a short sequence
query = "ATGCGTTTACGATCGATCGATCGATCGATCGATCG"

results = searcher.search_sequence(
    query,
    program="blastn",
    db="phages",
    max_hits=5,
    evalue=1e-5,
)

print(f"Found {len(results)} hits")
if not results.empty:
    results.head()

## Search a Protein Sequence

Search an unknown protein sequence against phage proteins.

In [ ]:
# Example protein sequence
protein_query = "MKTAYIAKQRQISFVKSHFSRQDILDLWIYHTQGYFP"

protein_results = searcher.search_sequence(
    protein_query,
    program="blastp",
    db="proteins",
    max_hits=10,
    evalue=1e-3,
)

print(f"Found {len(protein_results)} hits")
if not protein_results.empty:
    protein_results.head()

## Search a FASTA File

Search multiple sequences from a FASTA file.

In [ ]:
# Create a temporary FASTA file for demonstration
import tempfile
from pathlib import Path

fasta_content = ">seq1\nATGCGTTTACGATCGATCGATCG\n>seq2\nTTTAAACCCGGGTTTAAACCCGGG\n"
fasta_path = Path(tempfile.mktemp(suffix=".fasta"))
fasta_path.write_text(fasta_content)

# Search all sequences
batch_results = searcher.search_fasta(
    fasta_path,
    program="blastn",
    db="phages",
    max_hits=3,
)

print(f"Found {len(batch_results)} total hits")
if not batch_results.empty:
    batch_results.head(10)

# Clean up
fasta_path.unlink()

## BLAST Program Selection

| Program | Query Type | Database Type | Use Case |
|---------|-----------|---------------|----------|
| `blastn` | Nucleotide | Nucleotide | Find similar phage genomes |
| `blastp` | Protein | Protein | Find functional protein homologs |
| `blastx` | Nucleotide | Protein | Search translated DNA against proteins |
| `tblastn` | Protein | Nucleotide | Search protein against translated DNA |

## Search Host Genomes

Search against bacterial host genomes to find phage-host relationships.

In [ ]:
# Search against host genomes
host_results = searcher.search_sequence(
    query,
    program="blastn",
    db="hosts",
    max_hits=5,
)

print(f"Found {len(host_results)} hits against host genomes")
if not host_results.empty:
    host_results.head()

## Save Results to CSV

In [ ]:
if not results.empty:
    output_path = "blast_results.csv"
    results.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")

## Advanced: Using Extra BLAST Arguments

In [ ]:
# Example: use word size and dust filtering options
advanced_results = searcher.search_sequence(
    query,
    program="blastn",
    db="phages",
    max_hits=5,
    evalue=1e-10,
    extra_args=["-word_size", "11", "-dust", "no"],
)

print(f"Found {len(advanced_results)} hits with advanced parameters")